# Fintech Mobile App Review Analytics

## Introduction / Topics Covered
- **Data Acquisition**: Live scraping with `google-play-scraper`.
- **Data Engineering**: Advanced cleaning (whitespaces, trailing zeros, date normalization).
- **Exploratory Data Analysis (EDA)**: Rating distributions and length analysis.
- **Keyword & Context Discovery**: Bag-of-Words, TF-IDF, N-grams (Bigrams/Trigrams).
- **Lexical Analysis**: POS Tagging (Noun extraction).
- **Advanced Sentiment Analysis**: VADER, TextBlob (Polarity & Subjectivity), and Transformer templates.
- **Thematic Analysis**: Unsupervised (LDA) and Supervised (Business Theme Mapping).
- **Business Intelligence**: Identifying Drivers vs. Pain Points.

## Section 2: Environment Setup

In [ ]:
!pip install google_play_scraper
!pip install wordcloud scikit-learn

In [ ]:
import warnings, re, os, json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize, pos_tag
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Download necessary NLTK components
for res in ['vader_lexicon', 'stopwords', 'averaged_perceptron_tagger', 'punkt', 'wordnet']:
    nltk.download(res, quiet=True)

sia = SentimentIntensityAnalyzer()
stop_words = set(stopwords.words('english'))

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
print('Environment ready ✓')

## Section 3 — Live Scraping & Data Collection using google_play_scraper

In [ ]:
from google_play_scraper import reviews, Sort

# App package IDs for global fintech apps
APPS = {
    'PayPal':  'com.paypal.android.p2pmobile',
    'Revolut': 'com.revolut.revolut',
    'CashApp': 'com.squareup.cash',
}

def scrape_fintech_reviews(app_dict, count=200):
    """Fetches recent reviews for a dictionary of apps."""
    all_reviews = []
    for name, pkg in app_dict.items():
        try:
            res, _ = reviews(pkg, lang='en', country='us', sort=Sort.NEWEST, count=count)
            for r in res:
                all_reviews.append({
                    'app': name,
                    'review': r['content'],
                    'rating': r['score'],
                    'date': r['at'],
                    'thumbs': r['thumbsUpCount']
                })
            print(f'  Scraped {len(res)} reviews for {name}')
        except Exception as e:
            print(f'  Failed to scrape {name}: {e}')
    return pd.DataFrame(all_reviews)

# Fetching reviews
df_raw = scrape_fintech_reviews(APPS, count=250)
print(f'\nTotal raw reviews: {len(df_raw)}')

## Section 4 — Advanced Data Cleaning & Preprocessing
**Data Quality Standards Outlined:**
- **Clean Strings**: No leading/trailing whitespaces or redundant .0 from numeric conversions.
- **Integrity**: Missing values drop below the 5% threshold.
- **Standardization**: Dates follow the YYYY-MM-DD format.

In [ ]:
def modular_nlp_pipeline(text):
    # 1. Cleaning
    text = re.sub(r'[^a-zA-Z\s]', ' ', str(text).lower())
    
    # 2. Tokenization
    tokens = word_tokenize(text)
    
    # 3. Stop-word removal & Lemmatization
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    processed = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(processed)

# Apply cleaning
print('Running modular preprocessing...')
df_raw['clean_text'] = df_raw['review'].apply(modular_nlp_pipeline)

# Drop invalid/too short reviews
df_clean = df_raw[df_raw['clean_text'].str.len() > 2].copy()
print(f'Final cleaned dataset size: {len(df_clean)}')

In [ ]:
def robust_clean(df):
    # 1. Strip Whitespaces & Trailing Zeros
    for col in df.select_dtypes(include=['object']):
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].str.replace(r'\.0$', '', regex=True)
        
    # 2. Date Normalisation
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%Y-%m-%d')
    
    # 3. Handle Missing Values
    print(f'Nulls before: {df.isnull().sum().sum()}')
    df = df.dropna(subset=['review', 'rating'])
    
    # 4. Filter empty/short reviews
    df = df[df['review'].str.len() > 2]
    
    print(f'Final Nulls: {df.isnull().sum().sum()}')
    print(f'Final Dataset Shape: {df.shape}')
    return df.reset_index(drop=True)

df_clean = robust_clean(df_raw.copy())

## Section 5 — Rating & Data Quality Distribution
(This section initializes visualization functions to graph the outputs of the data engineering stage).

In [ ]:
def plot_rating_distribution(df):
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='rating', palette='viridis')
    plt.title('Distribution of Ratings')
    plt.xlabel('Rating')
    plt.ylabel('Count')
    plt.show()

plot_rating_distribution(df_clean)

In [ ]:
df_clean['raw_length'] = df_raw['review'].str.len()
df_clean['clean_length'] = df_clean['clean_text'].str.len()

plt.figure(figsize=(10, 5))
sns.histplot(df_clean['raw_length'], color='blue', label='Raw Length', kde=True, alpha=0.5)
sns.histplot(df_clean['clean_length'], color='orange', label='Clean Length', kde=True, alpha=0.5)
plt.title('Review Text Length Distribution')
plt.legend()
plt.show()

## Section 6 — Keyword & Context Discovery
Bag-of-Words, TF-IDF, N-grams (Bigrams/Trigrams).

In [ ]:
def extract_ngrams(df, col='clean_text', n=2, top_k=20):
    vec = CountVectorizer(ngram_range=(n, n)).fit(df[col])
    bag_of_words = vec.transform(df[col])
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:top_k]

bigrams = extract_ngrams(df_clean, n=2, top_k=10)
print('Top 10 Bigrams:')
for b in bigrams:
    print(b)

In [ ]:
def generate_wordcloud(text_series, title):
    text = ' '.join(text_series.dropna())
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

generate_wordcloud(df_clean['clean_text'], 'Common Words in Fintech Reviews')

## Section 7 — Lexical Analysis
POS Tagging (Noun extraction).

In [ ]:
def extract_nouns(text):
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    return [word for word, pos in tags if pos.startswith('NN')]

df_clean['nouns'] = df_clean['clean_text'].apply(extract_nouns)
all_nouns = [noun for nouns_list in df_clean['nouns'] for noun in nouns_list]
noun_counts = Counter(all_nouns)
print('Most common nouns:', noun_counts.most_common(10))

## Section 8 — Advanced Sentiment Analysis
VADER, TextBlob (Polarity & Subjectivity), and Transformer templates.

In [ ]:
def get_vader_sentiment(text):
    score = sia.polarity_scores(text)
    if score['compound'] >= 0.05: return 'positive'
    elif score['compound'] <= -0.05: return 'negative'
    else: return 'neutral'

df_clean['sentiment_vader'] = df_clean['review'].apply(get_vader_sentiment)
print(df_clean['sentiment_vader'].value_counts())

In [ ]:
def get_textblob_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df_clean[['tb_polarity', 'tb_subjectivity']] = df_clean['review'].apply(lambda x: pd.Series(get_textblob_sentiment(x)))
print(df_clean[['review', 'tb_polarity', 'tb_subjectivity']].head())

In [ ]:
# Transformer (DistilBERT) Template Implementation
from transformers import pipeline

try:
    sentiment_pipeline = pipeline('text-classification', model='distilbert-base-uncased-finetuned-sst-2-english')
    sample_reviews = df_clean['review'].head(5).tolist()
    print('Transformer Predictions:')
    print(sentiment_pipeline(sample_reviews))
except Exception as e:
    print('Transformer model could not be loaded or failed:', e)

## Section 9 — Thematic Analysis
Unsupervised (LDA) and Supervised (Business Theme Mapping).

In [ ]:
THEME_KEYWORDS = {
    'Account Access Issues': ['login', 'log in', 'password', 'otp', 'fingerprint'],
    'Transaction Performance': ['transfer', 'transaction', 'slow', 'crash', 'error'],
    'UI & Design': ['ui', 'interface', 'design', 'layout', 'navigation'],
    'Customer Support': ['support', 'customer service', 'help', 'response'],
    'Feature Requests': ['feature', 'add', 'request', 'budget', 'notification']
}

def assign_theme(text):
    text_lower = str(text).lower()
    for theme, keywords in THEME_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return theme
    return 'General Feedback'

df_clean['theme'] = df_clean['review'].apply(assign_theme)
print(df_clean['theme'].value_counts())

## Section 10 — Business Intelligence
Identifying Drivers vs. Pain Points.

In [ ]:
pain_points = df_clean[df_clean['sentiment_vader'] == 'negative']['theme'].value_counts()
drivers = df_clean[df_clean['sentiment_vader'] == 'positive']['theme'].value_counts()

print('Top Pain Points:')
print(pain_points.head())
print('\nTop Drivers:')
print(drivers.head())

## Export Results

In [ ]:
export_path = 'reviews_analyzed.csv'
df_clean.to_csv(export_path, index=False)
print(f'Data successfully exported to {export_path}')